In [2]:
import openai
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

In [3]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [4]:
query = "Can I get some card game?"

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [10]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="toys_and_games_1K",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


In [11]:
results = retrieve_data(query, k=20)
results

{'retrieved_context_ids': ['B0178G37NU',
  'B07P8JQYYN',
  'B09DFHRQPC',
  'B00IGR5ERI',
  'B0B1TVJ3GL',
  'B00LU1AUIK',
  'B0017LD7QU',
  'B00LBFNEB0',
  'B009UJB7WC',
  'B00BGCY6II',
  'B07BF5XF85',
  'B002B30FPY',
  'B0BHTS24SG',
  'B0062DVDUA',
  'B001CGOS86',
  'B00D8TT6Z0',
  'B08FWVD8RC',
  'B07PT3JZJ1',
  'B07CL2BSM8',
  'B0040HSUL0'],
 'retrieved_context': ['Lord of The Fries Super Deluxe Card Game Lightweight, funny card game: make combo meals from random ingredients Zombies! All new card art by Brian Snoddy Includes a complete Coffee Shop expansion Designed by award-winning game designer James Ernest All new card art by Brian Snoddy Lord of the Fries is the follow-up to Give Me the Brain, another game in the Fast Food Restaurant of the Damned. The mechanic is simple: combine the ingredients in your hand to build combo meals, with dishes like the Cowabunga, the Meat Munch, and of course the Lord of the Fries. This is the fourth edition of Lord of the Fries. It was introduced 

In [12]:
cohere_client = cohere.ClientV2()

In [13]:
to_rerank = results["retrieved_context"]

In [14]:
to_rerank

['Lord of The Fries Super Deluxe Card Game Lightweight, funny card game: make combo meals from random ingredients Zombies! All new card art by Brian Snoddy Includes a complete Coffee Shop expansion Designed by award-winning game designer James Ernest All new card art by Brian Snoddy Lord of the Fries is the follow-up to Give Me the Brain, another game in the Fast Food Restaurant of the Damned. The mechanic is simple: combine the ingredients in your hand to build combo meals, with dishes like the Cowabunga, the Meat Munch, and of course the Lord of the Fries. This is the fourth edition of Lord of the Fries. It was introduced in 1998 and has been updated and improved in every edition. The game returns to Cheapass Games after a brief vacation at Steve Jackson Games. This time, the game is designed to expand. The core box contains the original restaurant menu (55 cards) plus a completely new Coffee Shop menu, with all-new art and ingredients (also 55 cards). Each deck can support up to 6 p

In [15]:
query

'Can I get some card game?'

In [16]:
response = cohere_client.rerank(
    model="rerank-v5.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [17]:

response

V2RerankResponse(id='d4fca0e6-5640-462a-9f35-2d43d9115efa', results=[V2RerankResponseResultsItem(index=18, relevance_score=0.67857134, relevant_spans=None), V2RerankResponseResultsItem(index=3, relevance_score=0.6699124, relevant_spans=None), V2RerankResponseResultsItem(index=0, relevance_score=0.6413852, relevant_spans=None), V2RerankResponseResultsItem(index=16, relevance_score=0.6272842, relevant_spans=None), V2RerankResponseResultsItem(index=12, relevance_score=0.6072691, relevant_spans=None), V2RerankResponseResultsItem(index=19, relevance_score=0.5926125, relevant_spans=None), V2RerankResponseResultsItem(index=1, relevance_score=0.5840108, relevant_spans=None), V2RerankResponseResultsItem(index=14, relevance_score=0.56987864, relevant_spans=None), V2RerankResponseResultsItem(index=17, relevance_score=0.54844826, relevant_spans=None), V2RerankResponseResultsItem(index=7, relevance_score=0.5167388, relevant_spans=None), V2RerankResponseResultsItem(index=2, relevance_score=0.511727,

In [18]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [19]:
reranked_results

["OUR MOMENTS Kids: 100 Conversation Starter Cards for Families and Kids - Relationship Building - Car Travel, Road Trip & Card Game for Healthy Loving Family - Questions for Family Activities Quality time - for you and your child or children - Discover your children and let them get to know you. Educate through fun. Improve your listening skills - ask a question and listen carefully to the answer. Talk about it. There is no correct answer and no judgment. So many times parents are the last to know - It’s important that your child knows that whatever is happening in their life they can approach Mom or Dad. This instills confidence and self-esteem. They will learn this by the discussions you have around the questions in this game. Open the door and they will come to you. Good or bad - you need to be the FIRST to know. Get them talking and the rest will follow - This game is based on the principle of adult guided discussion. In a game setting, the child will be relaxed and receptive. Mak